<a href="https://www.kaggle.com/code/mrafraim/dl-day-45-intro-to-model-deployment?scriptVersionId=308906102" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Day 45: Intro to Model Deployment
*From Research to Production: Train → Save → Serve → Predict*

Welcome to Day 45!

What You’ll Learn Today:

1. Research vs Production mindset
2. What “serving a model” actually means
3. Full ML pipeline (end-to-end)
4. Saving & loading models
5. Basic inference pipeline
6. Real-world deployment architecture

If you found this notebook helpful, your **<b style="color:skyblue;">UPVOTE</b>** would be greatly appreciated! It helps others discover the work and supports continuous improvement.

---

# Mindset Shift: Research vs Production

Most learners stop after training.

Reality check:
- Research = you train and evaluate models in notebooks.
- Production = you make models *usable for real users reliably*.


| Aspect       | Research                         | Production                       |
|-------------|---------------------------------|----------------------------------|
| Focus       | Accuracy                        | Reliability & latency            |
| Environment | Notebook                        | API / Cloud / Server             |
| Inputs      | Clean datasets                  | User-provided, unpredictable    |
| Scale       | Single experiment               | 10,000+ concurrent users        |
| Goal        | Understand what works           | Deliver predictions consistently |


Research:

✔ “Model works on my notebook”

Production:

✔ “Model works for 10,000 users in real time”

# What “Serving a Model” Really Means

Serving = making the trained model accessible to external requests.

**Flow:** User → API → Model → Prediction → Response

- Input = anything user sends
- Model = trained CNN or ML model
- Output = prediction

Serving is not training.

- You do NOT compute gradients
- You disable randomness (dropout, batch norm in training mode)
- You must ensure consistency with training preprocessing

# Full End-to-End ML Pipeline

1. Train → validate → pick best model
2. Save model weights (torch.save)
3. Load model in service (torch.load)
4. Accept input (image, text, etc.)
5. Preprocess exactly as during training
6. Run inference (model.eval(), no_grad)
7. Return prediction to user
8. Monitor system performance


$$Train → Save → Deploy → Predict → Monitor$$

# Saving the Model

- Training is expensive
- You can’t retrain for every prediction
- Save weights once and reuse

**Rule**

`torch.save` saves only:
- Model parameters (weights & biases)
- NOT training state (optimizer, scheduler)

```python
torch.save(model.state_dict(), "model.pth")
```

# Loading the Model

To reuse model in deployment or inference.

### Critical Rule

Architecture MUST match saved model exactly.

Otherwise:
- Weights mismatch
- Model fails to load

```python
# Create model instance with SAME architecture
model = CNN()
model.load_state_dict(torch.load("model.pth"))
model.eval()  # important for inference
```

# Inference Pipeline (Real Use)

- Training = compute gradients, dropout active
- Inference = deterministic, fast, no gradients

### Practical Rules

1. Always call `model.eval()`
2. Use `torch.no_grad()` context
3. Preprocess input exactly as training

```python
def predict(image, model):
    """
    image: preprocessed tensor (batch_size=1)
    model: loaded PyTorch model
    """
    model.eval()
    with torch.no_grad():
        output = model(image)
        pred = output.argmax(dim=1)
    return pred
```

# Input Preprocessing (Often Overlooked)

Mismatch between training and inference preprocessing = garbage predictions

### Must Include

- Normalization
- Resizing / reshaping
- Tensor conversion
- Same channels/order as training

```python
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
```

# Minimal Deployment Architecture

## Simple local deployment

User → API (Flask/FastAPI) → Model → Prediction → Response

- API handles user requests
- Model predicts asynchronously
- Response returned to user

## Production-ready deployment

User  
↓  
Load Balancer → distributes traffic  
↓  
API Server → handles requests  
↓  
Model Server → runs model inference  
↓  
Prediction → returned to user

---
<p style="text-align:center; color:Red; font-size:18px;"> Deep Dive (Optional) </p>
A deployment system is not just “serving a model”.

It is a:
> Scalable, reliable, low-latency prediction system

Each component exists for:
- Speed
- Scalability
- Reliability

### 1. Minimal Local Deployment

#### Architecture

User → API → Model → Prediction → Response

#### Typical Stack

- API: FastAPI / Flask  
- Model: PyTorch  
- Environment: Single machine (CPU/GPU)

#### Request Flow

**1. User Request**
- Image / JSON input sent to API

**2. API Layer**
- Receives request
- Validates input
- Handles errors

**3. Preprocessing***
- Resize image
- Normalize
- Convert to tensor  

MUST match training preprocessing exactly

**4. Model Inference**

```python
model.eval()
with torch.no_grad():
    output = model(image)
    pred = output.argmax(dim=1)
```

**5. Response**

```json
{
  "prediction": "cat",
  "confidence": 0.92
}
```

#### Limitations

* Single point of failure
* Cannot handle high traffic
* Poor scalability
* Not fault tolerant

👉 Suitable only for:

* Testing
* Small apps
* Demos

### 2. Production-Ready Deployment

#### Architecture

User
↓
Load Balancer
↓
API Servers (Multiple)
↓
Model Servers (Multiple)
↓
Prediction → Response


#### Layer 1: Load Balancer

**Purpose:**

* Distribute traffic across servers
* Prevent overload
* Handle failures

**Tools:**

* NGINX
* AWS Elastic Load Balancer


#### Layer 2: API Servers

**Responsibilities:**

* Handle requests
* Validate input
* Preprocess data
* Route to model server
* Format response

**Key Rule:**

❗ Do NOT run heavy inference here

#### Layer 3: Model Servers

**Responsibilities:**

* Load trained model (once at startup)
* Run inference
* Return predictions

**Tools:**

* TorchServe
* TensorFlow Serving
* Custom microservice

#### Full Request Flow

1. User sends request
2. Load balancer routes request
3. API server processes input
4. API calls model server
5. Model server runs inference
6. Prediction returned to API
7. API sends response to user


### Critical Production Constraints

#### 1. Latency Budget

* Target: < 100ms
* Includes:

  * Network delay
  * Preprocessing
  * Inference

#### 2. Cold Start Problem

* Loading model per request = slow (seconds)

Solution:

* Load model once at startup

#### 3. Concurrency

* Handle multiple users
* Use async processing / batching

#### 4. Hardware Strategy

* API servers → CPU
* Model servers → GPU

### Advanced Enhancements

#### 1. Caching

* Store repeated predictions
* Reduce computation

#### 2. Task Queues

* Handle heavy workloads asynchronously

Tools:

* Redis
* Celery

#### 3. Monitoring

Track:

* Latency
* Errors
* Throughput

#### 4. Model Versioning

* Deploy multiple versions (v1, v2)
* A/B testing

---

# Things to Handle in Production

- Latency: must respond quickly
- Memory limits: large models may require batching
- Concurrency: multiple users at the same time
- Model versioning: update models without downtime
- Monitoring: track errors, drifts, failures

# Local Inference Example

Simulate serving for a single user locally.

Steps:
1. Take a sample image
2. Preprocess it
3. Send to loaded model
4. Print prediction

```python
from PIL import Image

image = Image.open("test.jpg")
image = transform(image).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model(image)
    pred = output.argmax(dim=1)

print("Predicted class:", pred.item())
```

# Key Takeaways from Day 45
- Training ≠ Deployment
- Serving = making predictions accessible reliably
- Save → Load → Predict = production core loop
- Preprocessing consistency is critical
- Production systems require reliability, scaling, monitoring

---
<p style="text-align:center; color:skyblue; font-size:18px;">
© 2026 Mostafizur Rahman
</p>
